<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>토크나이저 기본(Tokenizer Fundamentals)</strong>에 대해 학습합니다.</br></br>
>BPE 알고리즘의 원리를 이해하고, 실제 한국어 데이터셋(NSMC)으로 WordPiece 토크나이저를 학습한 뒤, 토큰 인코딩과 워드 임베딩까지 학습해봅시다.

<div style="text-align:center">

</div>

</br>

# 토크나이저 (Tokenizer)
> 토크나이저는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">텍스트를 모델이 처리할 수 있는 토큰 단위로 분리</mark>하는 전처리 도구입니다.

> 컴퓨터는 텍스트를 직접 이해하지 못합니다. 신경망은 행렬 연산만 수행하므로, 모든 입력은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">숫자(정수 또는 실수)</mark>여야 합니다.</br></br>
> 텍스트를 숫자로 변환하는 과정이 바로 토크나이징이며, 이것이 NLP 전처리의 첫 단계입니다. 텍스트는 컴퓨터 내부에서 유니코드 숫자의 나열로 저장되고, 모델이 알고 있는 토큰의 집합인 어휘 사전(Vocabulary)은 정수 인덱스로 관리됩니다.</br></br>
>그렇다면 텍스트를 어떤 단위로 쪼갤 것인가가 핵심 설계 결정입니다.</br></br>
 > <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단어 단위</mark>로 분리하면 "고양이가"와 "고양이는"을 다른 단어로 취급하여 어휘가 폭발적으로 커지고 미등록 단어(OOV)가 발생합니다.</br></br>
 > <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">문자 단위</mark>로 처리하면 OOV는 없지만 시퀀스가 지나치게 길어집니다.</br></br>
 > <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">서브워드 단위</mark>는 자주 나오는 조합은 하나의 토큰으로, 희귀한 단어는 쪼개서 처리하여 두 방식의 균형을 잡습니다.</br></br>
 > 현대 언어 모델(GPT, BERT, LLaMA)이 모두 서브워드 방식을 채택한 이유는 **어휘 크기와 표현력 사이의 최적 균형** 때문이며, 토크나이저의 설계가 바뀌면 같은 모델도 성능이 달라지므로 NLP 파이프라인에서 가장 근본적인 선택입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">방식</th>
      <th style="text-align:center">단위</th>
      <th>예시</th>
      <th>문제점</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">단어 기반</td><td style="text-align:center">단어</td><td><code>["안녕", "하세요"]</code></td><td>OOV (미등록 단어)</td></tr>
    <tr><td style="text-align:center">문자 기반</td><td style="text-align:center">글자</td><td><code>["안", "녕", "하"]</code></td><td>시퀀스 길이 폭증</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">서브워드 기반</mark></td><td style="text-align:center">서브워드</td><td><code>["안녕", "하", "세요"]</code></td><td>균형 잡힌 접근</td></tr>
  </tbody>
</table>

</br>

# BPE (Byte Pair Encoding)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">가장 빈번한 문자 쌍을 반복적으로 병합</mark>하여 서브워드 사전을 구축하는 알고리즘입니다.

<div style="text-align:center">

</div>

</br>

## get_stats 구현
> 현재 토큰 시퀀스에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">인접 토큰 쌍의 빈도</mark>를 수집합니다.

In [1]:
# TODO 1: BPE 실습에 사용할 테스트 vocab을 정의해봅시다.

vocab = {"l o w </w>": 5, "l o w e r </w>": 2, "n e w </w>": 6}
print(f"vocab: {vocab}")

vocab: {'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w </w>': 6}


In [2]:
# TODO 2: vocab 딕셔너리의 각 단어에서 인접 심볼 쌍의 빈도를 집계하여 반환하는 함수를 정의해봅시다.

def get_stats(vocab):
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] = pairs.get(pair, 0) + freq
    return pairs

In [3]:
# TODO 3: get_stats 함수를 실행하여 상위 3개 쌍을 출력해봅시다.

stats = get_stats(vocab)
print("빈도 상위 쌍:")
for pair, freq in sorted(stats.items(), key=lambda x: -x[1])[:3]:
    print(f"  {pair}: {freq}")

빈도 상위 쌍:
  ('w', '</w>'): 11
  ('l', 'o'): 7
  ('o', 'w'): 7


💡`</w>`란?
> `</w>`는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단어의 끝(End of Word)</mark>을 표시하는 특수 기호입니다.</br></br>
> BPE는 단어를 문자 단위로 쪼개서 시작하는데, 단어 경계를 구분하기 위해 끝에 `</w>`를 붙입니다.</br>
> ```
> low   → l o w </w>      ← "w 뒤에 단어가 끝남"
> lower → l o w e r </w>  ← "r 뒤에 단어가 끝남"
> ```
> 이렇게 하면 `w </w>`(단어 끝의 w)와 `w e`(뒤에 글자가 이어지는 w)를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다른 토큰으로 구분</mark>할 수 있어서, 디코딩 시 원래 단어 경계를 복원할 수 있습니다.

</br>

## BPE 전체 흐름

In [4]:
# TODO 4: 정규식 모듈을 활용하여, 주어진 쌍을 vocab에서 병합하는 함수를 정의해봅시다.

import re

def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in vocab:
        new_word = pattern.sub(''.join(pair), word)
        new_vocab[new_word] = vocab[word]
    return new_vocab

In [5]:
# TODO 5: 빈도 통계와 병합을 5회 반복 호출하여 BPE 병합 과정을 출력해봅시다.

vocab = {"l o w </w>": 5, "l o w e r </w>": 2, "n e w </w>": 6}

for i in range(5):
    pairs = get_stats(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Merge {i+1}: {best[0]} + {best[1]} → {''.join(best)}  (freq={pairs[best]})")

Merge 1: w + </w> → w</w>  (freq=11)
Merge 2: l + o → lo  (freq=7)
Merge 3: n + e → ne  (freq=6)
Merge 4: ne + w</w> → new</w>  (freq=6)
Merge 5: lo + w</w> → low</w>  (freq=5)


💡BPE의 장점
> 미등록 단어(OOV)를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">서브워드 조합으로 표현</mark>할 수 있습니다.</br>
> GPT, RoBERTa 등 대부분의 현대 LLM이 BPE 변형을 사용합니다.

💡사전 크기는 어떻게 정할까?
> 일반적으로 30,000~50,000 토큰이 많이 사용됩니다.</br>
> 사전이 크면 시퀀스가 짧아지고, 작으면 서브워드 분할이 세밀해집니다.

</br>

# 실전 토크나이저 학습
> 앞에서 BPE 알고리즘의 원리를 이해했으니, 이제 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실제 한국어 데이터셋</mark>으로 토크나이저를 학습해봅시다.</br></br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">NSMC(Naver Sentiment Movie Corpus)</mark> 데이터셋의 영화 리뷰 텍스트를 사용합니다.

</br>

## 데이터 준비
> NSMC는 네이버 영화 리뷰 20만 건으로 구성된 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">한국어 감성 분석 데이터셋</mark>입니다.

In [6]:
# TODO 6: NSMC 데이터셋을 다운로드해봅시다.

from pathlib import Path
from urllib.request import Request, urlopen

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt"
output_path = Path("ratings.txt")

if output_path.exists():
    print(f"이미 파일이 존재합니다: {output_path.resolve()}")
else:
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    response = urlopen(request)
    output_path.write_bytes(response.read())
    print(f"다운로드 완료: {output_path.resolve()}")

이미 파일이 존재합니다: /Users/sahong.pak/Documents/AI/My/Ch_2/Ch_2-1/정답/ratings.txt


In [7]:
# TODO 7: 다운로드한 데이터를 DataFrame으로 로드하고 구조를 확인해봅시다.

import pandas as pd

df = pd.read_table("ratings.txt", encoding="utf-8")
df = df.dropna(how="any")  # 결측값 제거

print(f"데이터 크기: {len(df):,}건")
print(f"컬럼: {list(df.columns)}")
df.head()

데이터 크기: 199,992건
컬럼: ['id', 'document', 'label']


,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


In [8]:
# TODO 8: document 컬럼의 텍스트를 파일로 저장해봅시다.

text_path = "naver_review.txt"
with open(text_path, "w", encoding="utf-8") as f:
    f.write("\n".join(df["document"].astype(str)))

print(f"저장 완료: {text_path} ({len(df):,}건)")

저장 완료: naver_review.txt (199,992건)


</br>

## WordPiece 토크나이저 학습
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">WordPiece</mark>는 BPE와 유사하지만, 빈도 대신 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">우도(likelihood)를 최대화</mark>하는 방향으로 병합합니다.</br></br>
> BERT 모델이 WordPiece 토크나이저를 사용하며, 서브워드 앞에 `##` 접두사를 붙여 구분합니다.</br>
> 예: `"토크나이저"` → `["토크", "##나이", "##저"]`

💡우도(Likelihood)란?
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">우도</mark>는 "현재 모델이 학습 데이터를 얼마나 잘 설명하는가"를 나타내는 확률값입니다.</br></br>
> **BPE**: "a"와 "b"가 100번 같이 나왔으니 병합 → <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">빈도</mark> 기준</br>
> **WordPiece**: "a"와 "b"를 병합하면 전체 텍스트의 확률이 더 높아지는가? → <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">우도</mark> 기준</br></br>
> 예를 들어, "불" + "고기"를 병합하면 "불고기"라는 의미 있는 단위가 되어 텍스트 설명력(우도)이 올라갑니다.</br>
> 반면 "기" + "는"은 자주 붙어 나오지만 의미 없는 조합이라 우도 향상이 적습니다.</br></br>
> 그래서 WordPiece는 단순히 많이 등장하는 쌍이 아니라, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">병합했을 때 가장 의미 있는 쌍</mark>을 선택합니다.

<div style="text-align:center">

</div>

In [9]:
# TODO 9: BertWordPieceTokenizer를 생성해봅시다.

from tokenizers import BertWordPieceTokenizer

tokenizer = BertWordPieceTokenizer(
    lowercase=False,     # 영어 대소문자 유지
    strip_accents=False,  # 한국어에는 악센트 없음
)
print(f"어휘 크기: {tokenizer.get_vocab_size()}")

어휘 크기: 0


In [10]:
# TODO 10: NSMC 텍스트로 WordPiece 토크나이저를 학습해봅시다.

tokenizer.train(
    files=["naver_review.txt"],
    vocab_size=30000,           # 어휘 사전 크기
    min_frequency=2,            # 최소 등장 빈도
    limit_alphabet=6000,        # 초기 알파벳 최대 수
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
    wordpieces_prefix="##",     # 서브워드 접두사
    show_progress=True,
)
print(f"학습 완료! 어휘 크기: {tokenizer.get_vocab_size():,}")




학습 완료! 어휘 크기: 30,000


💡WordPiece vs BPE
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">BPE</mark>: 가장 빈번한 쌍을 병합 (빈도 기반)</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">WordPiece</mark>: 병합 시 우도(likelihood)가 가장 높아지는 쌍을 선택</br></br>
> 두 알고리즘 모두 서브워드 사전을 구축하지만, WordPiece가 더 의미 있는 단위로 분할하는 경향이 있습니다.</br>
> GPT 계열은 BPE, BERT 계열은 WordPiece를 주로 사용합니다.

</br>

## 토큰 인코딩
> 학습된 토크나이저로 텍스트를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">토큰 → 정수 ID</mark>로 변환합니다.

In [11]:
# TODO 11: 흔한 문장과 희귀한 단어가 포함된 문장을 각각 인코딩하여 차이를 비교해봅시다.

# 흔한 단어로만 구성된 문장
text_common = "이 영화 정말 재미있었어요"
encoded_common = tokenizer.encode(text_common)
print(f"흔한 문장 토큰: {encoded_common.tokens}")
print(f"  → 흔한 단어는 통째로 하나의 토큰")

# 희귀한 단어가 포함된 문장
text_rare = "메타버스 블록체인 프롬프트엔지니어링"
encoded_rare = tokenizer.encode(text_rare)
print(f"\n희귀 단어 토큰: {encoded_rare.tokens}")
print(f"  → 사전에 없는 단어는 ##서브워드로 분할")

흔한 문장 토큰: ['이', '영화', '정말', '재미있었어요']
  → 흔한 단어는 통째로 하나의 토큰

희귀 단어 토큰: ['메', '##타', '##버스', '블록', '##체', '##인', '프', '##롬', '##프트', '##엔', '##지니', '##어', '##링']
  → 사전에 없는 단어는 ##서브워드로 분할


In [12]:
# TODO 12: 영어 텍스트도 인코딩하여 한국어와 비교해봅시다.

text_en = "I'm a student of SSAFY!"

encoded_en = tokenizer.encode(text_en)
print(f"토큰: {encoded_en.tokens}")
print(f"정수 ID: {encoded_en.ids}")

토큰: ['I', "'", 'm', 'a', 'st', '##ud', '##ent', 'of', 'S', '##S', '##A', '##F', '##Y', '!']
정수 ID: [45, 11, 81, 69, 15444, 24816, 16072, 10280, 55, 3731, 3991, 4226, 4561, 5]


💡특수 토큰 (Special Tokens)
> 토크나이저 학습 시 지정한 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">special_tokens</mark>는 모델이 텍스트 구조를 이해하기 위해 사용하는 예약된 토큰입니다.</br></br>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">토큰</th>
      <th>이름</th>
      <th>역할</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>[PAD]</code></td><td>Padding</td><td>길이가 다른 문장들을 같은 길이로 맞출 때 빈자리를 채움</td></tr>
    <tr><td style="text-align:center"><code>[UNK]</code></td><td>Unknown</td><td>사전에 없는 토큰을 대체 (서브워드로도 분해 불가능한 경우)</td></tr>
    <tr><td style="text-align:center"><code>[CLS]</code></td><td>Classification</td><td>문장의 시작을 표시, BERT에서 분류 작업 시 이 토큰의 벡터를 사용</td></tr>
    <tr><td style="text-align:center"><code>[SEP]</code></td><td>Separator</td><td>문장의 끝 또는 두 문장 사이의 구분을 표시</td></tr>
    <tr><td style="text-align:center"><code>[MASK]</code></td><td>Mask</td><td>BERT의 빈칸 맞추기(MLM) 학습 시 가려진 토큰을 표시</td></tr>
  </tbody>
</table>

> 예: `"영화가 좋았다"` → `[CLS] 영화 ##가 좋았 ##다 [SEP]`</br>
> 이 토큰들은 실제 텍스트가 아니라, 모델에게 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">문장의 구조적 정보</mark>를 알려주는 신호입니다.

💡토큰화 결과 해석
> 흔한 단어("영화", "재미있었어요")는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">통째로 하나의 토큰</mark>이 됩니다.</br>
> 사전에 없는 희귀한 단어만 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">##서브워드로 분할</mark>됩니다.</br></br>
> 예: `"프롬프트엔지니어링"` → `["프롬프트", "##엔지", "##니어", "##링"]`</br>
> `##` 접두사는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이전 토큰에 이어지는 서브워드</mark>라는 의미입니다.</br></br>
> 이것이 서브워드 토크나이저의 장점입니다: 흔한 단어는 빠르게 처리하고, 희귀한 단어도 서브워드 조합으로 표현할 수 있습니다.

</br>

## 워드 임베딩 (Word Embedding)
> 토큰 ID(정수)를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">고정 차원의 연속 벡터</mark>로 변환합니다.</br></br>
> 정수 ID 자체에는 의미 정보가 없습니다. `nn.Embedding`은 각 토큰 ID를 N차원 벡터로 매핑하여, 모델이 토큰 간 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">의미적 관계를 학습</mark>할 수 있게 합니다.

In [13]:
# TODO 13: nn.Embedding으로 임베딩 레이어를 생성하고 차원을 확인해봅시다.

import torch
import torch.nn as nn

vocab_size = tokenizer.get_vocab_size()
embedding_dim = 768  # BERT와 동일한 차원

embedding = nn.Embedding(vocab_size, embedding_dim)
print(f"임베딩 테이블: {embedding.weight.shape}")  # (vocab_size, 768)

임베딩 테이블: torch.Size([30000, 768])


In [14]:
# TODO 14: 특정 토큰의 임베딩 벡터를 조회해봅시다.

token_id = tokenizer.token_to_id("영화")
print(f"'영화' 토큰 ID: {token_id}")

input_id = torch.tensor([token_id], dtype=torch.long)
vector = embedding(input_id)
print(f"임베딩 벡터 shape: {vector.shape}")  # (1, 768)
print(f"벡터 값 (앞 5개): {vector.data[0, :5]}")

'영화' 토큰 ID: 5834
임베딩 벡터 shape: torch.Size([1, 768])
벡터 값 (앞 5개): tensor([-0.5965, -0.1683,  0.7466,  0.7479, -0.6753])


In [15]:
# TODO 15: 문장 전체를 임베딩 벡터로 변환해봅시다.

text = "이 영화 정말 재미있었어요!"
encoded = tokenizer.encode(text)

input_ids = torch.tensor(encoded.ids, dtype=torch.long)
sentence_vectors = embedding(input_ids)

print(f"토큰 수: {len(encoded.tokens)}")
print(f"임베딩 shape: {sentence_vectors.shape}")  # (토큰 수, 768)
for token, vec in zip(encoded.tokens, sentence_vectors):
    print(f"  {token:>10} → [{vec[0]:.4f}, {vec[1]:.4f}, ...]")  # 앞 2개 값

토큰 수: 5
임베딩 shape: torch.Size([5, 768])
           이 → [0.3626, 0.7971, ...]
          영화 → [-0.5965, -0.1683, ...]
          정말 → [0.1823, 1.4573, ...]
      재미있었어요 → [-1.1879, 0.0548, ...]
           ! → [-0.1932, -0.5926, ...]


💡nn.Embedding
> `nn.Embedding(num_embeddings, embedding_dim)`은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">룩업 테이블</mark>입니다.</br></br>
> 내부적으로 `(vocab_size, embedding_dim)` 크기의 가중치 행렬을 갖고, 정수 ID로 해당 행(row)을 조회합니다.</br>
> 학습 전에는 랜덤 벡터이지만, 모델 학습 과정에서 의미가 비슷한 토큰끼리 가까운 벡터를 갖도록 업데이트됩니다.</br></br>
> 이것이 바로 Ch_3에서 배운 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">임베딩 공간</mark>의 기초입니다.

In [16]:
# TODO 16: 학습된 토크나이저를 저장해봅시다. (다음 실습에서 사용합니다)

tokenizer.save_model(".", "naver_wp")
print("토크나이저 저장 완료: naver_wp-vocab.txt")

토크나이저 저장 완료: naver_wp-vocab.txt
